# Gerador de gráficos — EDA CAGED

Reconstrói os gráficos do notebook `EDA` (03_analytics) a partir de `workspace.silver.novo_caged_movimentacoes_semestral` e os exporta para `graficos/` (PNG estático via kaleido + HTML interativo). Também recalcula as tabelas de correlação da Seção 6.2 sobre a série mensal (bronze).

**Padrão visual:** todos os gráficos seguem o tema dos dashboards DemandA — paleta navy `#001236` (admissões/séries), teal `#009999` (salário/destaques), rosa `#D86DCD` (idade/saldo/alertas), linhas de referência em rosa escuro `#8f4f88`, fundo branco com gridlines sutis `#e6edf5`, tipografia Segoe UI e títulos navy alinhados à esquerda.

In [0]:
%pip install plotly statsmodels kaleido

In [0]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from scipy.stats import gaussian_kde
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pasta do relatório — escrita direta de arquivos de workspace (suportada em serverless)
BASE = "/Workspace/Users/kevyn.zap@hotmail.com/mba-novo-caged-data-engineering-free-edition/notebooks/03_analytics/EDA (1)/00_EDA_CAGED"
IMG_DIR = f"{BASE}/graficos"
os.makedirs(IMG_DIR, exist_ok=True)

# Mesmo carregamento do notebook EDA
df_semestral = spark.table("workspace.silver.novo_caged_movimentacoes_semestral").toPandas()
df_semestral['mediana_salario_admissao'] = df_semestral['mediana_salario_admissao'].astype('float64')
df_semestral['data_inicio'] = pd.to_datetime(df_semestral['data_inicio'])
df_semestral = df_semestral.sort_values('data_inicio').reset_index(drop=True)

admissoes = df_semestral['admissoes'].values
salarios = df_semestral['mediana_salario_admissao'].values
idades = df_semestral['mediana_idade_admissao'].values
desligamentos = df_semestral['desligamentos'].values
saldo = df_semestral['saldo'].values

# ------------------------------------------------------------------
# Tema visual DemandA — mesmo padrão dos dashboards HTML do projeto
# (navy #001236 · teal #009999 · rosa #D86DCD, fundo claro, grid sutil,
#  Segoe UI, títulos navy à esquerda, eixos cinza legíveis)
# ------------------------------------------------------------------
NAVY = '#001236'         # admissões / séries principais
TEAL = '#009999'         # salário / destaques
PINK = '#D86DCD'         # idade / saldo / alertas
TEAL_CLARO = '#5cc5c5'
ROSA_ESCURO = '#8f4f88'  # linhas de referência tracejadas (fit, refline)
CINZA = '#8a97a5'
GRID = '#e6edf5'         # gridlines sutis
BORDA = '#dce4ee'
TXT = '#5b6b7c'          # texto dos eixos
TINTA = '#1c2733'        # texto principal

def aplicar_tema(fig, altura=None):
    """Aplica o padrão visual dos dashboards DemandA a qualquer figura plotly."""
    upd = dict(
        font=dict(family='Segoe UI, system-ui, sans-serif', size=13, color=TXT),
        title=dict(font=dict(size=19, color=NAVY), x=0.01, xanchor='left'),
        paper_bgcolor='white', plot_bgcolor='white',
        margin=dict(l=64, r=30, t=64, b=52),
        hoverlabel=dict(bgcolor='white', bordercolor=BORDA, font=dict(color=TINTA)),
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color=TXT)),
    )
    if altura:
        upd['height'] = altura
    fig.update_layout(**upd)
    fig.update_annotations(font=dict(color=NAVY, size=13.5))
    fig.update_xaxes(showgrid=True, gridcolor=GRID, zeroline=False, linecolor=BORDA,
                     ticks='outside', tickcolor=BORDA, title_font=dict(color=TXT, size=12.5))
    fig.update_yaxes(showgrid=True, gridcolor=GRID, zeroline=True, zerolinecolor=BORDA,
                     linecolor=BORDA, ticks='outside', tickcolor=BORDA,
                     title_font=dict(color=TXT, size=12.5))
    return fig

def salvar(fig, nome):
    """Aplica o tema DemandA e exporta a figura como HTML interativo e PNG estático em graficos/."""
    fig = aplicar_tema(fig)
    html_path = f"{IMG_DIR}/{nome}.html"
    fig.write_html(html_path, include_plotlyjs='cdn')
    try:
        png_path = f"{IMG_DIR}/{nome}.png"
        fig.write_image(png_path, scale=2)
        print(f"✅ {nome}: HTML + PNG exportados")
    except Exception as e:
        print(f"⚠️ {nome}: HTML ok; PNG falhou ({type(e).__name__}): {str(e)[:300]}")

print(f"Dados carregados: {len(df_semestral)} semestres ({df_semestral['semestre'].iloc[0]} a {df_semestral['semestre'].iloc[-1]})")

In [0]:
# 2.1 Distribuição marginal: Admissões (idêntico ao notebook EDA)
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Histograma de Admissões", "Q-Q Plot (Normalidade)",
                    "Box-plot", "Densidade (KDE)"),
    specs=[[{"type": "histogram"}, {"type": "scatter"}],
           [{"type": "box"}, {"type": "scatter"}]]
)

fig.add_trace(
    go.Histogram(x=admissoes, nbinsx=8, name="Admissões", marker_color="#001236"),
    row=1, col=1
)

theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(admissoes)))
sample_quantiles = np.percentile(admissoes, np.linspace(1, 99, len(admissoes)))
fig.add_trace(
    go.Scatter(x=theoretical_quantiles, y=sample_quantiles, mode='markers',
               name='Dados', marker=dict(color='#001236', size=8)),
    row=1, col=2
)
x_line = np.linspace(theoretical_quantiles.min(), theoretical_quantiles.max(), 100)
y_line = stats.linregress(theoretical_quantiles, sample_quantiles)[0] * x_line + \
         stats.linregress(theoretical_quantiles, sample_quantiles)[1]
fig.add_trace(
    go.Scatter(x=x_line, y=y_line, mode='lines', name='Teórica',
               line=dict(color='#8f4f88', dash='dash')),
    row=1, col=2
)

fig.add_trace(
    go.Box(y=admissoes, name="Admissões", marker_color="#001236"),
    row=2, col=1
)

kde = gaussian_kde(admissoes)
x_kde = np.linspace(admissoes.min(), admissoes.max(), 200)
fig.add_trace(
    go.Scatter(x=x_kde, y=kde(x_kde), fill='tozeroy', name='Densidade KDE',
               line=dict(color='#001236')),
    row=2, col=2
)

fig.update_xaxes(title_text="Admissões", row=1, col=1)
fig.update_xaxes(title_text="Quantis Teóricos", row=1, col=2)
fig.update_xaxes(title_text="Densidade", row=2, col=2)
fig.update_yaxes(title_text="Frequência", row=1, col=1)
fig.update_yaxes(title_text="Quantis Amostra", row=1, col=2)
fig.update_yaxes(title_text="Admissões", row=2, col=1)
fig.update_yaxes(title_text="Densidade", row=2, col=2)

fig.update_layout(height=600, showlegend=True, title_text="Distribuição marginal — Admissões semestrais")
salvar(fig, "02_01_distribuicao_admissoes")

In [0]:
# 2.2 Distribuição marginal: Salário mediano de admissão
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Histograma de Salários", "Q-Q Plot (Normalidade)",
                    "Box-plot", "Densidade (KDE)"),
    specs=[[{"type": "histogram"}, {"type": "scatter"}],
           [{"type": "box"}, {"type": "scatter"}]]
)

fig.add_trace(
    go.Histogram(x=salarios, nbinsx=8, name="Salário", marker_color="#009999"),
    row=1, col=1
)

theoretical_quantiles_sal = stats.norm.ppf(np.linspace(0.01, 0.99, len(salarios)))
sample_quantiles_sal = np.percentile(salarios, np.linspace(1, 99, len(salarios)))
fig.add_trace(
    go.Scatter(x=theoretical_quantiles_sal, y=sample_quantiles_sal, mode='markers',
               name='Dados', marker=dict(color='#009999', size=8)),
    row=1, col=2
)
x_line_sal = np.linspace(theoretical_quantiles_sal.min(), theoretical_quantiles_sal.max(), 100)
y_line_sal = stats.linregress(theoretical_quantiles_sal, sample_quantiles_sal)[0] * x_line_sal + \
             stats.linregress(theoretical_quantiles_sal, sample_quantiles_sal)[1]
fig.add_trace(
    go.Scatter(x=x_line_sal, y=y_line_sal, mode='lines', name='Teórica',
               line=dict(color='#8f4f88', dash='dash')),
    row=1, col=2
)

fig.add_trace(
    go.Box(y=salarios, name="Salário", marker_color="#009999"),
    row=2, col=1
)

kde_sal = gaussian_kde(salarios)
x_kde_sal = np.linspace(salarios.min(), salarios.max(), 200)
fig.add_trace(
    go.Scatter(x=x_kde_sal, y=kde_sal(x_kde_sal), fill='tozeroy', name='Densidade KDE',
               line=dict(color='#009999')),
    row=2, col=2
)

fig.update_xaxes(title_text="Salário (R$)", row=1, col=1)
fig.update_xaxes(title_text="Quantis Teóricos", row=1, col=2)
fig.update_xaxes(title_text="Salário (R$)", row=2, col=2)
fig.update_yaxes(title_text="Frequência", row=1, col=1)
fig.update_yaxes(title_text="Quantis Amostra", row=1, col=2)
fig.update_yaxes(title_text="Salário (R$)", row=2, col=1)
fig.update_yaxes(title_text="Densidade", row=2, col=2)

fig.update_layout(height=600, showlegend=True, title_text="Distribuição marginal — Salário mediano de admissão")
salvar(fig, "02_02_distribuicao_salario")

In [0]:
# 2.3 Distribuição marginal: Idade mediana de admissão
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Histograma de Idades", "Q-Q Plot (Normalidade)",
                    "Box-plot", "Densidade (KDE)"),
    specs=[[{"type": "histogram"}, {"type": "scatter"}],
           [{"type": "box"}, {"type": "scatter"}]]
)

fig.add_trace(
    go.Histogram(x=idades, nbinsx=6, name="Idade", marker_color="#D86DCD"),
    row=1, col=1
)

theoretical_quantiles_age = stats.norm.ppf(np.linspace(0.01, 0.99, len(idades)))
sample_quantiles_age = np.percentile(idades, np.linspace(1, 99, len(idades)))
fig.add_trace(
    go.Scatter(x=theoretical_quantiles_age, y=sample_quantiles_age, mode='markers',
               name='Dados', marker=dict(color='#D86DCD', size=8)),
    row=1, col=2
)
x_line_age = np.linspace(theoretical_quantiles_age.min(), theoretical_quantiles_age.max(), 100)
y_line_age = stats.linregress(theoretical_quantiles_age, sample_quantiles_age)[0] * x_line_age + \
             stats.linregress(theoretical_quantiles_age, sample_quantiles_age)[1]
fig.add_trace(
    go.Scatter(x=x_line_age, y=y_line_age, mode='lines', name='Teórica',
               line=dict(color='#8f4f88', dash='dash')),
    row=1, col=2
)

fig.add_trace(
    go.Box(y=idades, name="Idade", marker_color="#D86DCD"),
    row=2, col=1
)

kde_age = gaussian_kde(idades)
x_kde_age = np.linspace(idades.min(), idades.max(), 200)
fig.add_trace(
    go.Scatter(x=x_kde_age, y=kde_age(x_kde_age), fill='tozeroy', name='Densidade KDE',
               line=dict(color='#D86DCD')),
    row=2, col=2
)

fig.update_xaxes(title_text="Idade (anos)", row=1, col=1)
fig.update_xaxes(title_text="Quantis Teóricos", row=1, col=2)
fig.update_xaxes(title_text="Idade (anos)", row=2, col=2)
fig.update_yaxes(title_text="Frequência", row=1, col=1)
fig.update_yaxes(title_text="Quantis Amostra", row=1, col=2)
fig.update_yaxes(title_text="Idade (anos)", row=2, col=1)
fig.update_yaxes(title_text="Densidade", row=2, col=2)

fig.update_layout(height=600, showlegend=True, title_text="Distribuição marginal — Idade mediana de admissão")
salvar(fig, "02_03_distribuicao_idade")

In [0]:
# 4.1 Relações bivariadas: Admissões × Desligamentos / Salário / Saldo
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Admissões vs Desligamentos", "Admissões vs Salário", "Admissões vs Saldo")
)

fig.add_trace(
    go.Scatter(x=admissoes, y=desligamentos, mode='markers+text',
               text=df_semestral['semestre'], textposition="top center",
               marker=dict(color='#001236', size=10),
               name='Dados'),
    row=1, col=1
)
z = np.polyfit(admissoes, desligamentos, 1)
p = np.poly1d(z)
fig.add_trace(
    go.Scatter(x=np.sort(admissoes), y=p(np.sort(admissoes)),
               mode='lines', name='Linear fit', line=dict(color='#8f4f88', dash='dash')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=admissoes, y=salarios, mode='markers+text',
               text=df_semestral['semestre'], textposition="top center",
               marker=dict(color='#009999', size=10),
               name='Dados'),
    row=1, col=2
)
z2 = np.polyfit(admissoes, salarios, 1)
p2 = np.poly1d(z2)
fig.add_trace(
    go.Scatter(x=np.sort(admissoes), y=p2(np.sort(admissoes)),
               mode='lines', name='Linear fit', line=dict(color='#8f4f88', dash='dash')),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=admissoes, y=saldo, mode='markers+text',
               text=df_semestral['semestre'], textposition="top center",
               marker=dict(color='#D86DCD', size=10),
               name='Dados'),
    row=1, col=3
)
z3 = np.polyfit(admissoes, saldo, 1)
p3 = np.poly1d(z3)
fig.add_trace(
    go.Scatter(x=np.sort(admissoes), y=p3(np.sort(admissoes)),
               mode='lines', name='Linear fit', line=dict(color='#8f4f88', dash='dash')),
    row=1, col=3
)

fig.update_xaxes(title_text="Admissões", row=1, col=1)
fig.update_xaxes(title_text="Admissões", row=1, col=2)
fig.update_xaxes(title_text="Admissões", row=1, col=3)
fig.update_yaxes(title_text="Desligamentos", row=1, col=1)
fig.update_yaxes(title_text="Salário (R$)", row=1, col=2)
fig.update_yaxes(title_text="Saldo", row=1, col=3)

fig.update_layout(height=400, showlegend=False, title_text="Relações bivariadas — correlações com admissões")
salvar(fig, "04_01_correlacoes_bivariadas")

In [0]:
# 5.1 Decomposição STL da série semestral de admissões (period=2)
from statsmodels.tsa.seasonal import seasonal_decompose

ts_admissoes = df_semestral.set_index('data_inicio')['admissoes']
decomposition = seasonal_decompose(ts_admissoes, model='additive', period=2, extrapolate_trend='freq')

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=("Série Original", "Tendência", "Sazonalidade", "Resíduos"),
    shared_xaxes=True
)
fig.add_trace(go.Scatter(x=ts_admissoes.index, y=ts_admissoes.values, name='Original',
                         mode='lines+markers', line=dict(color='#001236')), row=1, col=1)
fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend.values,
                         name='Tendência', mode='lines', line=dict(color='#D86DCD', width=3)), row=2, col=1)
fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal.values,
                         name='Sazonalidade', mode='lines', line=dict(color='#009999')), row=3, col=1)
fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid.values,
                         name='Resíduos', mode='lines', line=dict(color='#8a97a5')), row=4, col=1)

fig.update_yaxes(title_text="Admissões", row=1, col=1)
fig.update_yaxes(title_text="Tendência", row=2, col=1)
fig.update_yaxes(title_text="Sazonal", row=3, col=1)
fig.update_yaxes(title_text="Resíduos", row=4, col=1)

fig.update_layout(height=800, showlegend=True, title_text="Decomposição STL — série de admissões")
salvar(fig, "05_01_decomposicao_stl")

print("✅ Decomposição STL concluída")
print(f"   Tendência explicada: {(1 - decomposition.resid.var() / ts_admissoes.var()):.2%}")

In [0]:
# 5.2 ACF/PACF e teste ADF (idêntico ao notebook EDA)
from statsmodels.tsa.stattools import adfuller, acf, pacf

adf_result = adfuller(admissoes, maxlag=1)
print("📉 TESTE DE ESTACIONARIDADE (ADF)")
print(f"   Estatística ADF: {adf_result[0]:.4f}")
print(f"   p-value: {adf_result[1]:.4f}")
print(f"   Conclusão: {'Série estacionária ✅' if adf_result[1] < 0.05 else '⚠️ Série não-estacionária'}")

fig = make_subplots(rows=2, cols=1, subplot_titles=("ACF (Autocorrelation)", "PACF (Partial Autocorrelation)"))

n_lags_acf = min(5, len(admissoes) - 1)
n_lags_pacf = min(4, len(admissoes) // 2 - 1)
acf_data = acf(admissoes, nlags=n_lags_acf)
pacf_data = pacf(admissoes, nlags=n_lags_pacf, method='ywm')

fig.add_trace(go.Bar(x=list(range(len(acf_data))), y=acf_data, name="ACF", marker_color='#001236'), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(len(pacf_data))), y=pacf_data, name="PACF", marker_color='#009999'), row=2, col=1)

fig.update_layout(height=600, showlegend=True, title_text="Autocorrelação (ACF) e correlação parcial (PACF)")
salvar(fig, "05_02_acf_pacf")

In [0]:
# 6.2 Tabelas de correlação features × alvo (t e t+1) — série mensal CAGED
# (reproduz a célula corrigida 6.1/6.2 do notebook EDA para recuperar os nomes das features)
df_mensal = spark.sql("""
  SELECT
    `_competencia_referencia` AS competencia,
    to_date(concat(`_competencia_referencia`, '01'), 'yyyyMMdd') AS data_mes,
    sum(CASE WHEN `tipomovimentação` IN ('10','20','25','35','70','97')
             THEN 1 ELSE 0 END) AS admissoes,
    sum(CASE WHEN `tipomovimentação` IN ('31','32','33','40','43','45','50','60','80','90','98')
             THEN 1 ELSE 0 END) AS desligamentos,
    median(CASE WHEN `tipomovimentação` IN ('10','20','25','35','70','97')
                THEN cast(replace(`salário`, ',', '.') AS DOUBLE) END) AS mediana_salario
  FROM workspace.bronze.novo_caged_movimentacoes
  GROUP BY 1, 2
  ORDER BY 1
""").toPandas().sort_values('data_mes').reset_index(drop=True)
df_mensal['data_mes'] = pd.to_datetime(df_mensal['data_mes'])

df_ml = df_mensal.copy()
for lag in [1, 3, 6, 12]:
    df_ml[f'adm_lag_{lag}m'] = df_ml['admissoes'].shift(lag)
df_ml['adm_media_3m']  = df_ml['admissoes'].shift(1).rolling(3).mean()
df_ml['adm_media_12m'] = df_ml['admissoes'].shift(1).rolling(12).mean()
df_ml['adm_std_3m']    = df_ml['admissoes'].shift(1).rolling(3).std()
df_ml['mes_sin'] = np.sin(2 * np.pi * df_mensal['data_mes'].dt.month / 12)
df_ml['mes_cos'] = np.cos(2 * np.pi * df_mensal['data_mes'].dt.month / 12)
df_ml['adm_yoy'] = df_ml['admissoes'] / df_ml['adm_lag_12m'] - 1
df_ml['desl_lag_1m']    = df_ml['desligamentos'].shift(1)
df_ml['saldo_lag_1m']   = (df_ml['admissoes'] - df_ml['desligamentos']).shift(1)
df_ml['salario_lag_1m'] = df_ml['mediana_salario'].shift(1)
df_ml['t'] = np.arange(len(df_ml))
df_ml['alvo_t1'] = df_ml['admissoes'].shift(-1)

feature_cols = [c for c in df_ml.columns
                if c.startswith(('adm_', 'mes_', 'desl_', 'saldo_', 'salario_', 't'))
                and c not in ('alvo_t1',)]

def corr_table(target):
    rows = {}
    for c in feature_cols:
        valid = df_ml[[c, target]].dropna()
        if len(valid) > 2:
            r, p = stats.pearsonr(valid[c], valid[target])
            rows[c] = (r, p)
    return pd.DataFrame(rows, index=['pearson_r', 'p_value']).T.sort_values('pearson_r', ascending=False)

print(f"Série mensal: {len(df_ml)} meses ({df_ml['competencia'].min()} a {df_ml['competencia'].max()})\n")
print("Correlação com ADMISSÕES (t):")
print(corr_table('admissoes').round(4).to_string())
print("\nCorrelação com ADMISSÕES (t+1) — poder preditivo real:")
print(corr_table('alvo_t1').round(4).to_string())

In [0]:
# PNG estáticos via matplotlib (fallback: kaleido não tem Chrome no serverless)
# Reproduz os mesmos 6 gráficos como PNG para embutir no EDA_caged.md
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil

# --- Tema DemandA (mesmo padrão dos dashboards HTML do projeto) ---
plt.rcParams.update({
    'text.color': '#5b6b7c', 'axes.labelcolor': '#5b6b7c',
    'axes.edgecolor': '#dce4ee', 'axes.linewidth': 1.0,
    'xtick.color': '#5b6b7c', 'ytick.color': '#5b6b7c',
    'axes.grid': True, 'grid.color': '#e6edf5', 'grid.linewidth': 0.8,
    'axes.facecolor': 'white', 'figure.facecolor': 'white',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlecolor': '#001236',
    'figure.titlesize': 13, 'figure.titleweight': 'bold',
})

S1 = df_semestral['semestre'].astype(str).tolist()

def salvar_png(fig_mpl, nome):
    tmp = f"/tmp/{nome}.png"
    fig_mpl.savefig(tmp, dpi=150, bbox_inches='tight')
    plt.close(fig_mpl)
    shutil.copy(tmp, f"{IMG_DIR}/{nome}.png")
    print(f"✅ {nome}.png (matplotlib)")

# --- 2.1 Admissões ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Distribuição marginal — Admissões semestrais", fontsize=13, fontweight='bold', color='#001236')
axes[0, 0].hist(admissoes, bins=8, color='#001236', edgecolor='white')
axes[0, 0].set_title("Histograma de Admissões"); axes[0, 0].set_xlabel("Admissões"); axes[0, 0].set_ylabel("Frequência")
lr = stats.linregress(theoretical_quantiles, sample_quantiles)
xs = np.linspace(theoretical_quantiles.min(), theoretical_quantiles.max(), 100)
axes[0, 1].scatter(theoretical_quantiles, sample_quantiles, color='#001236', s=40)
axes[0, 1].plot(xs, lr.slope * xs + lr.intercept, '--', color='#8f4f88')
axes[0, 1].set_title("Q-Q Plot (Normalidade)"); axes[0, 1].set_xlabel("Quantis Teóricos"); axes[0, 1].set_ylabel("Quantis Amostra")
axes[1, 0].boxplot(admissoes, vert=True)
axes[1, 0].set_title("Box-plot"); axes[1, 0].set_ylabel("Admissões")
axes[1, 1].fill_between(x_kde, kde(x_kde), color='#001236', alpha=0.5)
axes[1, 1].plot(x_kde, kde(x_kde), color='#001236')
axes[1, 1].set_title("Densidade (KDE)"); axes[1, 1].set_xlabel("Admissões"); axes[1, 1].set_ylabel("Densidade")
plt.tight_layout(rect=[0, 0, 1, 0.95])
salvar_png(fig, "02_01_distribuicao_admissoes")

# --- 2.2 Salário ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Distribuição marginal — Salário mediano de admissão", fontsize=13, fontweight='bold', color='#001236')
axes[0, 0].hist(salarios, bins=8, color='#009999', edgecolor='white')
axes[0, 0].set_title("Histograma de Salários"); axes[0, 0].set_xlabel("Salário (R$)"); axes[0, 0].set_ylabel("Frequência")
lr_s = stats.linregress(theoretical_quantiles_sal, sample_quantiles_sal)
xs_s = np.linspace(theoretical_quantiles_sal.min(), theoretical_quantiles_sal.max(), 100)
axes[0, 1].scatter(theoretical_quantiles_sal, sample_quantiles_sal, color='#009999', s=40)
axes[0, 1].plot(xs_s, lr_s.slope * xs_s + lr_s.intercept, '--', color='#8f4f88')
axes[0, 1].set_title("Q-Q Plot (Normalidade)"); axes[0, 1].set_xlabel("Quantis Teóricos"); axes[0, 1].set_ylabel("Quantis Amostra")
axes[1, 0].boxplot(salarios, vert=True)
axes[1, 0].set_title("Box-plot"); axes[1, 0].set_ylabel("Salário (R$)")
axes[1, 1].fill_between(x_kde_sal, kde_sal(x_kde_sal), color='#009999', alpha=0.5)
axes[1, 1].plot(x_kde_sal, kde_sal(x_kde_sal), color='#009999')
axes[1, 1].set_title("Densidade (KDE)"); axes[1, 1].set_xlabel("Salário (R$)"); axes[1, 1].set_ylabel("Densidade")
plt.tight_layout(rect=[0, 0, 1, 0.95])
salvar_png(fig, "02_02_distribuicao_salario")

# --- 2.3 Idade ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Distribuição marginal — Idade mediana de admissão", fontsize=13, fontweight='bold', color='#001236')
axes[0, 0].hist(idades, bins=6, color='#D86DCD', edgecolor='white')
axes[0, 0].set_title("Histograma de Idades"); axes[0, 0].set_xlabel("Idade (anos)"); axes[0, 0].set_ylabel("Frequência")
lr_a = stats.linregress(theoretical_quantiles_age, sample_quantiles_age)
xs_a = np.linspace(theoretical_quantiles_age.min(), theoretical_quantiles_age.max(), 100)
axes[0, 1].scatter(theoretical_quantiles_age, sample_quantiles_age, color='#D86DCD', s=40)
axes[0, 1].plot(xs_a, lr_a.slope * xs_a + lr_a.intercept, '--', color='#8f4f88')
axes[0, 1].set_title("Q-Q Plot (Normalidade)"); axes[0, 1].set_xlabel("Quantis Teóricos"); axes[0, 1].set_ylabel("Quantis Amostra")
axes[1, 0].boxplot(idades, vert=True)
axes[1, 0].set_title("Box-plot"); axes[1, 0].set_ylabel("Idade (anos)")
axes[1, 1].fill_between(x_kde_age, kde_age(x_kde_age), color='#D86DCD', alpha=0.5)
axes[1, 1].plot(x_kde_age, kde_age(x_kde_age), color='#D86DCD')
axes[1, 1].set_title("Densidade (KDE)"); axes[1, 1].set_xlabel("Idade (anos)"); axes[1, 1].set_ylabel("Densidade")
plt.tight_layout(rect=[0, 0, 1, 0.95])
salvar_png(fig, "02_03_distribuicao_idade")

# --- 4.1 Bivariadas ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle("Relações bivariadas — correlações com admissões", fontsize=13, fontweight='bold', color='#001236')
pares = [(desligamentos, "Desligamentos", p, '#001236'), (salarios, "Salário (R$)", p2, '#009999'), (saldo, "Saldo", p3, '#D86DCD')]
for ax, (yv, ylab, fit, cor) in zip(axes, pares):
    ax.scatter(admissoes, yv, color=cor, s=45, zorder=3)
    xs_f = np.sort(admissoes)
    ax.plot(xs_f, fit(xs_f), '--', color='#8f4f88')
    for xi, yi, s in zip(admissoes, yv, S1):
        ax.annotate(s, (xi, yi), fontsize=7, xytext=(3, 4), textcoords='offset points')
    ax.set_xlabel("Admissões"); ax.set_ylabel(ylab)
axes[0].set_title("Admissões vs Desligamentos")
axes[1].set_title("Admissões vs Salário")
axes[2].set_title("Admissões vs Saldo")
plt.tight_layout(rect=[0, 0, 1, 0.93])
salvar_png(fig, "04_01_correlacoes_bivariadas")

# --- 5.1 STL ---
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Decomposição STL — série de admissões", fontsize=13, fontweight='bold', color='#001236')
axes[0].plot(ts_admissoes.index, ts_admissoes.values, '-o', color='#001236', markersize=4); axes[0].set_ylabel("Admissões")
axes[1].plot(decomposition.trend.index, decomposition.trend.values, color='#D86DCD', linewidth=2.5); axes[1].set_ylabel("Tendência")
axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values, color='#009999'); axes[2].set_ylabel("Sazonal")
axes[3].plot(decomposition.resid.index, decomposition.resid.values, color='#8a97a5'); axes[3].set_ylabel("Resíduos")
plt.tight_layout(rect=[0, 0, 1, 0.95])
salvar_png(fig, "05_01_decomposicao_stl")

# --- 5.2 ACF/PACF ---
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle("Autocorrelação (ACF) e correlação parcial (PACF)", fontsize=13, fontweight='bold', color='#001236')
axes[0].bar(range(len(acf_data)), acf_data, color='#001236', edgecolor='white')
axes[0].axhline(0, color='#5b6b7c', linewidth=0.8)
axes[0].set_title("ACF (Autocorrelation)"); axes[0].set_ylabel("Correlação")
axes[1].bar(range(len(pacf_data)), pacf_data, color='#009999', edgecolor='white')
axes[1].axhline(0, color='#5b6b7c', linewidth=0.8)
axes[1].set_title("PACF (Partial Autocorrelation)"); axes[1].set_xlabel("Lag"); axes[1].set_ylabel("Correlação parcial")
plt.tight_layout(rect=[0, 0, 1, 0.94])
salvar_png(fig, "05_02_acf_pacf")

print("\n🎉 PNGs estáticos e HTMLs interativos concluídos em graficos/")

In [0]:
# 6.1 Gráfico da série mensal de admissões (2021-01 a 2026-08) + média móvel 12m
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil

# Tema DemandA (garante o padrão mesmo executando esta célula isolada)
plt.rcParams.update({
    'text.color': '#5b6b7c', 'axes.labelcolor': '#5b6b7c',
    'axes.edgecolor': '#dce4ee', 'axes.linewidth': 1.0,
    'xtick.color': '#5b6b7c', 'ytick.color': '#5b6b7c',
    'axes.grid': True, 'grid.color': '#e6edf5', 'grid.linewidth': 0.8,
    'axes.facecolor': 'white', 'figure.facecolor': 'white',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlecolor': '#001236',
    'figure.titlesize': 13, 'figure.titleweight': 'bold',
})

mm12 = df_mensal.set_index('data_mes')['admissoes'].rolling(12).mean()

# --- plotly (HTML interativo) ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_mensal['data_mes'], y=df_mensal['admissoes'],
                          mode='lines+markers', name='Admissões (mensal)',
                          line=dict(color='#001236'), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=mm12.index, y=mm12.values, mode='lines',
                          name='Média móvel 12m', line=dict(color='#009999', width=3)))
fig.update_layout(height=500, showlegend=True,
                  title_text="Série mensal de admissões — CAGED (jan/2021 a ago/2026)",
                  xaxis_title="Mês", yaxis_title="Admissões")
salvar(fig, "06_01_serie_mensal")

# --- matplotlib (PNG estático) ---
fig_m, ax = plt.subplots(figsize=(14, 6))
fig_m.suptitle("Série mensal de admissões — CAGED (jan/2021 a ago/2026)", fontsize=13, fontweight='bold', color='#001236')
ax.plot(df_mensal['data_mes'], df_mensal['admissoes'], '-o', color='#001236', markersize=4, label='Admissões (mensal)')
ax.plot(mm12.index, mm12.values, color='#009999', linewidth=2.5, label='Média móvel 12m')
ax.set_xlabel("Mês")
ax.set_ylabel("Admissões")
ax.legend()
plt.tight_layout(rect=[0, 0, 1, 0.94])
png_tmp = "/tmp/06_01_serie_mensal.png"
fig_m.savefig(png_tmp, dpi=150, bbox_inches='tight')
plt.close(fig_m)
shutil.copy(png_tmp, f"{IMG_DIR}/06_01_serie_mensal.png")
print(f"✅ 06_01_serie_mensal.png ({len(df_mensal)} meses)")